# GPU 아키텍처와 트랜스포머 연산 - GPU 병렬 처리와 행렬 연산

- Tutorial ID: `cs-2-1`
- Tutorial: GPU 아키텍처와 트랜스포머 연산
- Section ID: `cs-2-1-1`
- Section: GPU 병렬 처리와 행렬 연산


# 061 | GPU 아키텍처와 트랜스포머 연산
## GPU 병렬 처리와 행렬 연산

> **대상**: 딥러닝이 왜 GPU에서 빠른지 궁금한 분, 트랜스포머 연산 최적화에 관심 있는 분  
> **목표**: GPU의 병렬 처리 원리를 이해하고, 행렬 연산이 GPU에서 어떻게 실행되는지 파악한다.

---

## 📌 목차
1. CPU vs GPU: 무엇이 다른가?
2. GPU 아키텍처: SM, CUDA Core, Warp
3. GEMM: 행렬 곱셈의 GPU 최적화
4. 트랜스포머 연산의 병렬화 분석
5. GPU 메모리 계층: HBM, SRAM, 레지스터
6. Flash Attention: 메모리 효율적 Attention
7. 배치 크기와 GPU 활용률
8. 실습: PyTorch GPU 프로파일링 (CPU 시뮬레이션)

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib
from functools import partial
import warnings
warnings.filterwarnings('ignore')

# Colab/로컬/브라우저 환경에서 한글 폰트가 없을 때 자동으로 NanumGothic을 등록합니다.
# (Colab 기본 런타임에는 NanumGothic이 없어서 matplotlib 한글 glyph 경고가 발생할 수 있습니다.)
import os
import urllib.request
import matplotlib.font_manager as fm

font_path = '/tmp/NanumGothic-Regular.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve(
            'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf',
            font_path,
        )
    fm.fontManager.addfont(font_path)
    matplotlib.rcParams['font.family'] = ['NanumGothic', 'DejaVu Sans']
except Exception as e:
    print(f'한글 폰트 설정 경고: {e}')

plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=3, suppress=True)

# PyTorch 가용성 확인
try:
    import torch
    TORCH_AVAILABLE = True
    CUDA_AVAILABLE = torch.cuda.is_available()
    print(f'PyTorch {torch.__version__} 사용 가능!')
    print(f'CUDA 사용 가능: {CUDA_AVAILABLE}')
    if CUDA_AVAILABLE:
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    else:
        print('→ CPU 환경에서 실습합니다 (결과는 GPU 환경과 다를 수 있음)')
except ImportError:
    TORCH_AVAILABLE = False
    CUDA_AVAILABLE = False
    print('PyTorch 미설치 — NumPy로 시뮬레이션합니다.')
    print('설치: pip install torch')

print('\n라이브러리 로드 완료!')

---
## 1. CPU vs GPU: 무엇이 다른가?

```
┌─────────────────────────────────────────────────────┐
│  CPU (Intel i9)              GPU (NVIDIA A100)      │
│  ┌─────────────┐             ┌──────────────────┐   │
│  │ 강력한 코어 8개│             │ 작은 코어 6912개 │   │
│  │ 복잡한 제어장치│             │ 단순한 제어장치  │   │
│  │ 큰 캐시      │             │ 작은 캐시        │   │
│  │ 높은 클럭    │             │ 낮은 클럭        │   │
│  └─────────────┘             └──────────────────┘   │
│                                                     │
│  복잡한 순차 작업에 강    단순한 병렬 작업에 강         │
│  OS, 게임 로직, DB       행렬 곱셈, 딥러닝 학습        │
└─────────────────────────────────────────────────────┘
```

**왜 딥러닝은 GPU가 유리한가?**  
행렬 곱셈 C = A × B에서:
- C[i,j] = A[i,:] · B[:,j] (내적)
- 각 (i,j) 원소는 **독립적으로 계산 가능** → 병렬 처리!
- 1000×1000 행렬: 1,000,000개 내적을 동시에 계산 가능

In [ ]:
# ============================================================
# CPU vs GPU 개념 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# CPU 다이어그램
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('CPU 아키텍처\n(적은 수의 강력한 코어)', fontsize=13)

# CPU 코어 (2x4 배치)
core_positions_cpu = [(1+i*2.2, 4+j*2.5) for j in range(2) for i in range(4)]
colors_cpu = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'] * 2
for idx, ((x, y), color) in enumerate(zip(core_positions_cpu, colors_cpu)):
    rect = patches.FancyBboxPatch((x-0.8, y-0.9), 1.6, 1.8,
                                   boxstyle='round,pad=0.1',
                                   facecolor=color, edgecolor='black', alpha=0.8)
    ax.add_patch(rect)
    # 각 코어 내부에 L1 캐시 표시
    rect2 = patches.FancyBboxPatch((x-0.5, y+0.1), 1.0, 0.5,
                                    boxstyle='round,pad=0.05',
                                    facecolor='yellow', edgecolor='black', alpha=0.9)
    ax.add_patch(rect2)
    ax.text(x, y-0.3, f'코어{idx+1}', ha='center', va='center', fontsize=9, color='white', weight='bold')
    ax.text(x, y+0.35, 'L1$', ha='center', va='center', fontsize=7)

# L3 캐시
rect_l3 = patches.FancyBboxPatch((0.2, 2.0), 9.6, 1.2,
                                   boxstyle='round,pad=0.1',
                                   facecolor='#FFF176', edgecolor='black', alpha=0.8)
ax.add_patch(rect_l3)
ax.text(5, 2.6, 'L3 캐시 (공유, 크고 느림)', ha='center', va='center', fontsize=10)

# RAM
rect_ram = patches.FancyBboxPatch((0.2, 0.2), 9.6, 1.2,
                                   boxstyle='round,pad=0.1',
                                   facecolor='#BDBDBD', edgecolor='black', alpha=0.8)
ax.add_patch(rect_ram)
ax.text(5, 0.8, 'RAM (주메모리, ~32GB)', ha='center', va='center', fontsize=10)
ax.axis('off')

# GPU 다이어그램
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_title('GPU 아키텍처\n(많은 수의 작은 CUDA Core)', fontsize=13)

# SM (Streaming Multiprocessor) 표시
sm_colors = ['#FF5252', '#FF7043', '#FF8F00', '#FDD835',
             '#66BB6A', '#26C6DA', '#42A5F5', '#7E57C2',
             '#EC407A', '#78909C', '#26A69A', '#8D6E63']
sm_positions = [(0.5+i*3.0, 8.5-j*2.8) for j in range(4) for i in range(3)]

for idx, ((x, y), color) in enumerate(zip(sm_positions[:12], sm_colors)):
    rect = patches.FancyBboxPatch((x-0.35, y-1.0), 2.4, 1.8,
                                   boxstyle='round,pad=0.05',
                                   facecolor=color, edgecolor='black', alpha=0.7)
    ax2.add_patch(rect)
    ax2.text(x+0.85, y-0.1, f'SM {idx+1}\n(수백개\nCUDA Core)', 
             ha='center', va='center', fontsize=6.5, color='white', weight='bold')

# HBM (GPU 메모리)
rect_hbm = patches.FancyBboxPatch((0.1, 0.2), 9.8, 0.7,
                                   boxstyle='round,pad=0.1',
                                   facecolor='#1565C0', edgecolor='black', alpha=0.8)
ax2.add_patch(rect_hbm)
ax2.text(5, 0.55, 'HBM (고대역폭 메모리, A100: 80GB, ~2TB/s)', 
         ha='center', va='center', fontsize=9, color='white', weight='bold')
ax2.axis('off')

plt.suptitle('CPU vs GPU 아키텍처 비교', fontsize=14)
plt.tight_layout()
plt.show()

print('▶ CPU: 코어 수 적지만 각각 강력 → 복잡한 순차 로직에 유리')
print('▶ GPU: 코어 수 많고 단순 → 같은 연산을 대량으로 병렬 처리에 유리')
print('       예: A100 = 6912개 CUDA Core (FP32 기준)')

---
## 2. GPU 아키텍처: SM, CUDA Core, Warp

### GPU의 계층 구조

```
GPU 전체
  └─ SM (Streaming Multiprocessor) × 108개 (A100 기준)
       └─ Warp × 4개 (동시 실행 단위)
            └─ CUDA Core × 32개 (한 Warp = 32 스레드)
                 └─ 레지스터 (각 스레드의 로컬 변수)
```

### Warp이란?
- GPU는 **32개의 스레드**를 하나의 Warp으로 묶어 동시에 실행
- 모든 스레드가 **같은 명령어**를 실행 (SIMT: Single Instruction, Multiple Thread)
- 행렬 곱셈에서 각 스레드가 하나의 C[i,j] 계산 담당

### Tensor Core
- NVIDIA Volta (2017)부터 도입된 특수 하드웨어
- 4×4 행렬 곱셈을 단 1 사이클에 수행!
- FP16, BF16, TF32에서 최고 성능
- A100: 312 TFLOPS (Tensor Core, BF16 기준)

In [ ]:
# ============================================================
# GEMM (행렬 곱셈) 병렬화 시뮬레이션

In [ ]:
print('=== 행렬 곱셈(GEMM)의 병렬화 원리 ===')
print()

# C = A @ B
# C[i,j] = sum_k A[i,k] * B[k,j]
# → 각 (i,j) 쌍은 독립적으로 계산 가능!

M, N, K = 4, 4, 4  # 작은 예제
A = np.random.randint(1, 5, (M, K))
B = np.random.randint(1, 5, (K, N))

print(f'A ({M}×{K}):')
print(A)
print(f'\nB ({K}×{N}):')
print(B)

print(f'\nC = A @ B → 각 C[i,j]는 독립적으로 계산 가능!')
print(f'총 {M*N}개의 독립적인 내적 연산 → {M*N}개 스레드로 병렬 처리')
print()

# 순차적 계산 (CPU 스타일)
C_sequential = np.zeros((M, N), dtype=int)
operations = []  # 각 원소 계산 과정 기록

for i in range(M):
    for j in range(N):
        # 이 계산이 GPU에서는 하나의 스레드가 담당
        val = sum(A[i, k] * B[k, j] for k in range(K))
        C_sequential[i, j] = val
        operations.append((i, j, val))

print('C = A @ B 결과:')
print(C_sequential)
print(f'\n검증 (np 결과): {np.allclose(C_sequential, A @ B)}')
print()
print('GPU에서는 이 16개 원소를 동시에 계산!')
print(f'→ M={M}, N={N}이면 {M*N}개 스레드 사용')
print(f'→ 실제 GPT-3: d_model=12288, 입력 토큰=2048')
print(f'  Q = X @ W_Q: (2048×12288) @ (12288×12288) → C 크기: 2048×12288 = {2048*12288:,}개 원소 병렬 계산!')

In [ ]:
# ============================================================
# 타일링(Tiling): GPU가 캐시를 활용하는 방법

In [ ]:
print('=== 타일링(Tiling) 기법 ===')
print()
print('문제: 큰 행렬은 GPU 메모리(SRAM)에 한번에 안 들어감')
print('해결: 행렬을 작은 타일로 나눠서 처리')
print()

def matmul_naive(A, B):
    """단순 행렬 곱셈 (비효율적)"""
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    C = np.zeros((M, N))
    for i in range(M):
        for j in range(N):
            for k in range(K):
                C[i, j] += A[i, k] * B[k, j]
    return C

def matmul_tiled(A, B, tile_size=2):
    """
    타일링 기반 행렬 곱셈
    
    핵심 아이디어:
    - 행렬을 tile_size × tile_size 블록으로 나눔
    - 각 블록은 공유 메모리(SRAM)에 올려서 재사용
    - 메모리 접근 횟수를 크게 줄임
    """
    M, K = A.shape
    K2, N = B.shape
    C = np.zeros((M, N))
    
    # 타일 단위로 반복
    for i0 in range(0, M, tile_size):
        for j0 in range(0, N, tile_size):
            for k0 in range(0, K, tile_size):
                # 타일 범위 계산
                i_end = min(i0 + tile_size, M)
                j_end = min(j0 + tile_size, N)
                k_end = min(k0 + tile_size, K)
                
                # 타일을 '공유 메모리'로 로드 (시뮬레이션)
                A_tile = A[i0:i_end, k0:k_end]  # GPU SRAM에 올림
                B_tile = B[k0:k_end, j0:j_end]  # GPU SRAM에 올림
                
                # 타일 내부에서 계산 (빠른 SRAM 사용)
                C[i0:i_end, j0:j_end] += A_tile @ B_tile
    
    return C

# 테스트
A_test = np.random.randn(8, 8).astype(np.float32)
B_test = np.random.randn(8, 8).astype(np.float32)

C_ref = A_test @ B_test
C_tiled = matmul_tiled(A_test, B_test, tile_size=4)

print(f'타일링 결과 검증: {np.allclose(C_ref, C_tiled)}')
print()
print('타일링의 장점:')
print('  타일 크기 T × T 일 때, 메모리 접근 횟수:')
print('  - Naive: M × N × 2K 번 (A, B 모두 K번씩 읽음)')
print('  - Tiled: M × N × 2K / T 번 (타일을 재사용!)')
print(f'  - T=32 (Warp 크기)이면 ~32배 메모리 접근 감소!')

---
## 3. 트랜스포머 연산의 병렬화 분석

In [ ]:
# ============================================================
# 트랜스포머 각 연산의 병렬화 가능성 분석

In [ ]:
print('=== 트랜스포머 연산별 병렬화 분석 ===')
print()

# GPT-2 Medium 기준
B = 1     # 배치 크기
T = 1024  # 시퀀스 길이
D = 1024  # d_model
H = 16    # 헤드 수
dk = D // H  # 헤드당 차원 = 64
D_ff = 4 * D  # FFN 중간 차원 = 4096

operations = [
    ('Q = X @ W_Q', T * D, D * D, 'GEMM: 완벽한 병렬화 가능'),
    ('K = X @ W_K', T * D, D * D, 'GEMM: 완벽한 병렬화 가능'),
    ('V = X @ W_V', T * D, D * D, 'GEMM: 완벽한 병렬화 가능'),
    ('Q @ K.T 점수', T * T * H, 0, 'GEMM: 헤드별 병렬화 가능 (T×T 행렬)'),
    ('Softmax(score)', T * T * H, 0, '행 방향 의존성: 각 행은 독립적으로 가능'),
    ('Attn @ V', T * dk * H, 0, 'GEMM: 헤드별 병렬화 가능'),
    ('W_O 투영', T * D, D * D, 'GEMM: 완벽한 병렬화 가능'),
    ('FFN Linear1', T * D_ff, D * D_ff, 'GEMM: 완벽한 병렬화 가능'),
    ('GELU 활성화', T * D_ff, 0, '원소별 연산: 완벽한 병렬화 가능'),
    ('FFN Linear2', T * D, D_ff * D, 'GEMM: 완벽한 병렬화 가능'),
    ('Layer Norm', T * D, 0, '각 토큰 독립 (내부 리덕션은 순차)'),
    ('Residual Add', T * D, 0, '원소별: 완벽한 병렬화 가능'),
]

print(f'GPT-2 Medium 기준: B={B}, T={T}, D={D}, H={H}, d_k={dk}')
print(f'{'연산':<20} {'출력 크기':>15} {'병렬 처리'}')
print('-' * 70)

total_elements = 0
for op_name, out_elem, param_elem, parallel_note in operations:
    total_elements += out_elem
    print(f'{op_name:<20} {out_elem:>10,} 원소  {parallel_note}')

print('-' * 70)
print(f'총 계산 원소: {total_elements:,}')
print()
print('▶ 대부분의 연산이 완벽하게 병렬화 가능 → GPU가 매우 효율적!')
print('▶ Softmax의 분모 계산(sum)이 유일한 순차 의존성')
print('  → 하지만 각 행은 독립적이므로 행 단위로 병렬화 가능')

In [ ]:
# ============================================================
# 행렬 크기에 따른 GPU 활용률 (이론적 분석)

In [ ]:
print('=== 행렬 크기와 GPU 활용률 ===')
print()

# NVIDIA A100 SXM4 스펙
GPU_SPECS = {
    'A100': {
        'peak_tflops_fp16': 312,   # TFLOPS (Tensor Core)
        'peak_tflops_fp32': 77.6,  # TFLOPS
        'hbm_bandwidth': 2.0,      # TB/s
        'hbm_size': 80,            # GB
        'sm_count': 108,
        'cuda_cores': 6912,
        'tensor_cores': 432,
    },
    'V100': {
        'peak_tflops_fp16': 125,
        'peak_tflops_fp32': 15.7,
        'hbm_bandwidth': 0.9,
        'hbm_size': 32,
        'sm_count': 80,
        'cuda_cores': 5120,
        'tensor_cores': 640,
    },
    'RTX 4090': {
        'peak_tflops_fp16': 165,
        'peak_tflops_fp32': 82.6,
        'hbm_bandwidth': 1.008,
        'hbm_size': 24,
        'sm_count': 128,
        'cuda_cores': 16384,
        'tensor_cores': 512,
    }
}

print(f'{'GPU':<12} {'FP16 성능':>12} {'FP32 성능':>12} {'메모리 대역폭':>14} {'VRAM':>8}')
print('-' * 62)
for gpu_name, specs in GPU_SPECS.items():
    print(f'{gpu_name:<12} {specs["peak_tflops_fp16"]:>10} TFLOPS {specs["peak_tflops_fp32"]:>9} TFLOPS {specs["hbm_bandwidth"]:>10} TB/s {specs["hbm_size"]:>5} GB')

print()
print('산술 강도 (Arithmetic Intensity) 분석:')
print('  AI = FLOPs / Memory Access (bytes)')
print('  AI가 높을수록 메모리보다 연산이 병목 → GPU 효율 높음')
print()

# 행렬 곱셈 M×K @ K×N의 산술 강도
def arithmetic_intensity_gemm(M, K, N, dtype_bytes=2):  # FP16=2bytes
    flops = 2 * M * K * N  # 곱셈 + 덧셈
    mem_bytes = (M * K + K * N + M * N) * dtype_bytes  # A, B, C 읽기/쓰기
    return flops / mem_bytes, flops, mem_bytes

sizes = [(64, 4096, 4096), (512, 4096, 4096), (2048, 4096, 4096)]
print(f'{'행렬 크기 (M,K,N)':<25} {'FLOPs':>15} {'메모리(MB)':>12} {'산술강도':>10}')
print('-' * 65)
for M, K, N in sizes:
    ai, flops, mem = arithmetic_intensity_gemm(M, K, N)
    print(f'({M:4d},{K:5d},{N:5d}){"":>3} {flops/1e9:>12.1f} G {mem/1e6:>10.1f} MB {ai:>10.1f}')

print()
print('A100 기준 Ridge Point (연산 최적 AI):')
ridge = GPU_SPECS["A100"]['peak_tflops_fp16'] * 1e12 / (GPU_SPECS["A100"]['hbm_bandwidth'] * 1e12)
print(f'  Ridge Point = {GPU_SPECS["A100"]["peak_tflops_fp16"]} TFLOPS / {GPU_SPECS["A100"]["hbm_bandwidth"]} TB/s = {ridge:.0f}')
print(f'  AI > {ridge:.0f}: 연산 제한(compute-bound) → GPU 완전 활용')
print(f'  AI < {ridge:.0f}: 메모리 대역폭 제한(memory-bound) → 메모리가 병목')

---
## 4. Flash Attention: 메모리 효율적 Attention

### 표준 Attention의 문제점

```
표준 Attention: S = Q @ Kᵀ  →  HBM에 S 저장 (T×T 크기!)
                P = softmax(S)  →  HBM에서 S 읽기, P 저장
                O = P @ V       →  HBM에서 P 읽기

T=2048, H=16이면: S 크기 = 16 × 2048 × 2048 × 2 bytes = 128 MB
→ 시퀀스가 길어지면 T² 메모리 필요!
```

### Flash Attention의 해결책

```
S 행렬을 HBM에 저장하지 않고, 타일 단위로 처리
  → HBM 읽기/쓰기 횟수를 O(T²) → O(T) 로 줄임!
  → 메모리 절약 + 속도 향상
```

In [ ]:
# ============================================================
# Flash Attention vs 표준 Attention 메모리 비교

In [ ]:
print('=== Flash Attention 메모리 절감 분석 ===')
print()

def standard_attention_memory(B, H, T, d_k, dtype_bytes=2):
    """
    표준 Attention이 HBM에 저장해야 하는 메모리 계산
    """
    Q_size = B * H * T * d_k * dtype_bytes
    K_size = B * H * T * d_k * dtype_bytes
    V_size = B * H * T * d_k * dtype_bytes
    S_size = B * H * T * T * dtype_bytes   # 문제! T² 크기
    P_size = B * H * T * T * dtype_bytes   # softmax 결과도 T²
    O_size = B * H * T * d_k * dtype_bytes
    
    total = Q_size + K_size + V_size + S_size + P_size + O_size
    return {
        'Q': Q_size, 'K': K_size, 'V': V_size,
        'S (score)': S_size, 'P (softmax)': P_size, 'O': O_size,
        'total': total
    }

def flash_attention_memory(B, H, T, d_k, block_size=64, dtype_bytes=2):
    """
    Flash Attention의 메모리 계산
    (S, P를 HBM에 저장하지 않고 블록 단위 처리)
    """
    Q_size = B * H * T * d_k * dtype_bytes
    K_size = B * H * T * d_k * dtype_bytes
    V_size = B * H * T * d_k * dtype_bytes
    O_size = B * H * T * d_k * dtype_bytes
    # S, P는 블록 크기만큼만 SRAM에 보유 (HBM 저장 안 함)
    S_block = B * H * block_size * block_size * dtype_bytes
    
    total = Q_size + K_size + V_size + O_size + S_block
    return {
        'Q': Q_size, 'K': K_size, 'V': V_size,
        'S_block (SRAM)': S_block, 'O': O_size,
        'total': total
    }

# 다양한 시퀀스 길이에 대한 메모리 비교
B, H, d_k = 1, 12, 64
seq_lengths = [512, 1024, 2048, 4096, 8192]

print(f'B={B}, H={H}, d_k={d_k}, dtype=FP16(2bytes)')
print()
print(f'{'시퀀스 길이':>12} {'표준 Attention':>16} {'Flash Attention':>16} {'절감 비율':>10}')
print('-' * 58)

std_mems, flash_mems = [], []
for T in seq_lengths:
    std = standard_attention_memory(B, H, T, d_k)
    flash = flash_attention_memory(B, H, T, d_k)
    
    std_mb = std['total'] / 1e6
    flash_mb = flash['total'] / 1e6
    saving = (1 - flash_mb / std_mb) * 100
    
    std_mems.append(std_mb)
    flash_mems.append(flash_mb)
    
    print(f'T={T:5d}        {std_mb:>12.1f} MB  {flash_mb:>12.1f} MB  {saving:>9.1f}%')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(seq_lengths, std_mems, 'r-o', label='표준 Attention (O(T²))', linewidth=2, markersize=8)
axes[0].plot(seq_lengths, flash_mems, 'b-s', label='Flash Attention (O(T))', linewidth=2, markersize=8)
axes[0].set_xlabel('시퀀스 길이 T')
axes[0].set_ylabel('HBM 메모리 사용량 (MB)')
axes[0].set_title('표준 vs Flash Attention\n메모리 사용량 비교')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

savings = [(s-f)/s*100 for s, f in zip(std_mems, flash_mems)]
axes[1].bar(range(len(seq_lengths)), savings, color=['#4CAF50' if s > 50 else '#FF9800' for s in savings])
axes[1].set_xlabel('시퀀스 길이')
axes[1].set_ylabel('메모리 절감률 (%)')
axes[1].set_title('Flash Attention 메모리 절감률')
axes[1].set_xticks(range(len(seq_lengths)))
axes[1].set_xticklabels([str(t) for t in seq_lengths])
axes[1].grid(True, alpha=0.3, axis='y')
for i, s in enumerate(savings):
    axes[1].text(i, s+0.5, f'{s:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print('\n▶ 시퀀스가 길어질수록 Flash Attention의 절감 효과가 커집니다!')
print('▶ T=8192이면 표준 Attention은 거의 불가능 (메모리 부족)')

In [ ]:
# ============================================================
# Flash Attention 알고리즘 (개념적 구현)

In [ ]:
def flash_attention_forward(Q, K, V, block_size=4):
    """
    Flash Attention 순전파 (개념적 구현, Dao et al. 2022)
    
    핵심 아이디어:
    1. Q를 블록으로 나눔 (Q_1, Q_2, ...)
    2. 각 Q 블록에 대해 K, V를 순회하며 Attention 계산
    3. Online softmax trick으로 전체 S를 저장하지 않고 계산
    
    Args:
        Q, K, V: (T, d_k) 행렬
        block_size: 타일 크기
    Returns:
        O: (T, d_k) 출력
    """
    T, d_k = Q.shape
    scale = 1.0 / np.sqrt(d_k)
    O = np.zeros_like(Q)
    
    # Q 블록별 반복
    for i_start in range(0, T, block_size):
        i_end = min(i_start + block_size, T)
        Q_block = Q[i_start:i_end]  # (block, d_k) → SRAM에 올라옴
        
        # Online softmax를 위한 누적값
        m_i = np.full(i_end - i_start, -np.inf)  # 최대값 (수치 안정성)
        l_i = np.zeros(i_end - i_start)           # 정규화 합
        O_block = np.zeros((i_end - i_start, d_k))
        
        # K, V 블록별 반복 (전체 K, V를 순회)
        for j_start in range(0, T, block_size):
            j_end = min(j_start + block_size, T)
            K_block = K[j_start:j_end]  # (block, d_k) → SRAM에 올라옴
            V_block = V[j_start:j_end]  # (block, d_k) → SRAM에 올라옴
            
            # 이 블록의 점수 계산 (SRAM 내에서, HBM 접근 없음!)
            S_block = Q_block @ K_block.T * scale  # (block, block)
            
            # Online softmax trick:
            # 새 최대값을 보면서 이전 누적값을 재조정
            m_new = np.maximum(m_i, S_block.max(axis=1))
            
            # exp(S - m_new): 안전한 softmax
            exp_S = np.exp(S_block - m_new[:, np.newaxis])
            
            # 이전 누적값 재조정 (m이 바뀌었으므로)
            correction = np.exp(m_i - m_new)
            l_i = correction * l_i + exp_S.sum(axis=1)
            O_block = correction[:, np.newaxis] * O_block + exp_S @ V_block
            m_i = m_new
        
        # 최종 정규화
        O[i_start:i_end] = O_block / l_i[:, np.newaxis]
    
    return O


def standard_attention(Q, K, V):
    """표준 Attention (비교용)"""
    scale = 1.0 / np.sqrt(Q.shape[-1])
    S = Q @ K.T * scale
    # Softmax
    S -= S.max(axis=-1, keepdims=True)
    P = np.exp(S)
    P /= P.sum(axis=-1, keepdims=True)
    return P @ V


# 검증
np.random.seed(42)
T, d_k = 16, 8
Q = np.random.randn(T, d_k).astype(np.float64)
K = np.random.randn(T, d_k).astype(np.float64)
V = np.random.randn(T, d_k).astype(np.float64)

O_standard = standard_attention(Q, K, V)
O_flash = flash_attention_forward(Q, K, V, block_size=4)

print('Flash Attention 구현 검증:')
print(f'표준 Attention shape: {O_standard.shape}')
print(f'Flash Attention shape: {O_flash.shape}')
print(f'결과 동일? {np.allclose(O_standard, O_flash, atol=1e-6)}')
print(f'최대 오차: {np.max(np.abs(O_standard - O_flash)):.2e}')
print()
print('▶ Flash Attention은 표준 Attention과 수학적으로 동일한 결과를 줍니다!')
print('▶ 차이는 중간 행렬(S, P)을 HBM에 저장하지 않는다는 것뿐!')
print('▶ 실제 구현(Dao et al.)은 CUDA 커널로 작성되어 훨씬 빠름')

In [ ]:
# ============================================================
# GPU 활용을 위한 배치 크기 최적화

In [ ]:
print('=== 배치 크기와 GPU 활용률 관계 ===')
print()
print('GPU의 병렬 처리 단위: Warp (32 스레드)')
print('배치 크기가 너무 작으면 GPU가 놀게 됩니다!')
print()

# NumPy로 배치 크기에 따른 성능 시뮬레이션
D = 512
W = np.random.randn(D, D).astype(np.float32)
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]

throughputs = []
latencies = []

for B in batch_sizes:
    X = np.random.randn(B, D).astype(np.float32)
    
    # 워밍업
    _ = X @ W
    
    n = max(5, 100 // B)
    t = time.perf_counter()
    for _ in range(n):
        Y = X @ W
    elapsed = (time.perf_counter() - t) / n * 1000  # ms
    
    throughput = B / elapsed * 1000  # samples/sec
    latencies.append(elapsed)
    throughputs.append(throughput)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(batch_sizes, latencies, 'r-o', linewidth=2, markersize=8)
axes[0].set_xlabel('배치 크기')
axes[0].set_ylabel('지연 시간 (ms)')
axes[0].set_title('배치 크기 vs 지연 시간\n(낮을수록 좋음 — 개별 요청 처리 속도)')
axes[0].set_xscale('log', base=2)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(batch_sizes)
axes[0].set_xticklabels(batch_sizes)

axes[1].plot(batch_sizes, throughputs, 'b-s', linewidth=2, markersize=8)
axes[1].set_xlabel('배치 크기')
axes[1].set_ylabel('처리량 (samples/sec)')
axes[1].set_title('배치 크기 vs 처리량\n(높을수록 좋음 — 단위 시간 내 처리량)')
axes[1].set_xscale('log', base=2)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(batch_sizes)
axes[1].set_xticklabels(batch_sizes)

plt.suptitle(f'배치 크기에 따른 성능 (D={D} 행렬 곱셈, NumPy 시뮬레이션)', fontsize=12)
plt.tight_layout()
plt.show()

print('▶ 처리량: 배치 클수록 증가 (GPU 병렬화 효율 증가)')
print('▶ 지연시간: 배치 클수록 증가 (하나의 요청이 더 오래 걸림)')
print('▶ 실제 서비스에서는 지연시간 vs 처리량 트레이드오프 고려')
print('▶ 학습에서는 보통 가능한 한 큰 배치 (메모리 허용 범위 내)')

---
## 📚 핵심 정리

| 개념 | 설명 | 트랜스포머 관련성 |
|------|------|------------------|
| **CUDA Core** | GPU의 기본 계산 단위 | 행렬 원소 병렬 계산 |
| **Tensor Core** | 4×4 행렬 곱 전용 하드웨어 | GEMM 가속 (FP16/BF16) |
| **Warp** | 32 스레드 동시 실행 단위 | 행렬 행/열 블록 단위 처리 |
| **HBM** | GPU 메인 메모리 (넓은 대역폭) | Q, K, V 행렬 저장 |
| **SRAM** | SM 내 공유 메모리 (빠름, 작음) | Flash Attention 타일 저장 |
| **타일링** | 행렬을 작은 블록으로 나눠 처리 | GEMM 최적화의 핵심 |
| **Flash Attention** | S, P를 HBM에 저장 안 함 | 메모리 O(T²)→O(T) |

### 다음 단계
- **062**: FP16/BF16 혼합 정밀도 학습
- Flash Attention 논문: Dao et al. "FlashAttention" (NeurIPS 2022)